In [ ]:
!pip install llama-index==0.12.46
!pip install llama-index-embeddings-huggingface==0.5.5
!pip install peft==0.15.2
!pip install auto-gptq==0.7.1
!pip install optimum==1.26.1+
!pip install bitsandbytes==0.46.1

ERROR: Invalid requirement: 'optimum==1.26.1+': Expected end or semicolon (after version specifier)
    optimum==1.26.1+
           ~~~~~~~~^


In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor

In [ ]:
# import any embedding model on HF hub (https://huggingface.co/spaces/mteb/leaderboard)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
# Settings.embed_model = HuggingFaceEmbedding(model_name="thenlper/gte-large") # alternative model

Settings.llm = None
Settings.chunk_size = 256
Settings.chunk_overlap = 25

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


LLM is explicitly disabled. Using MockLLM.


In [ ]:
documents = SimpleDirectoryReader("articles").load_data()

In [ ]:
import re

filters = [
    "page_label:",
    "file_path:",
    "context information is below",
    "https://medium.com",
    "am",
    "pm",
    "written by",
    "followers",
    "no responses yet",
    "what are your thoughts",
    "kubernetes kubernetes cluster",
    "", "",  # Special characters
]

def clean_text(text):
    lines = text.splitlines()
    cleaned_lines = []
    for line in lines:
        line_lower = line.lower()
        if any(f in line_lower for f in filters):
            continue
        if re.search(r"\d{1,2}/\d{1,2}/\d{2,4}", line):  # Remove date-like strings
            continue
        if re.search(r"https?://\S+", line):  # Remove URLs
            continue
        if not line.strip():  # Skip empty lines
            continue
        cleaned_lines.append(line.strip())
    return "\n".join(cleaned_lines)


# Example record
sample = {
    "instruction": "What should I do before upgrading Kubernetes to a new version?",
    "output": """Context information is below.
---------------------
page_label: 4
file_path: /content/articles/Top 10 Kubernetes Issues and Solutions_ Overcoming Common Challenges in Container Orchestration _ by Prateek Malhotra _ Medium.pdf

10. Upgrades and Compatibility:
Issue: Performing upgrades of Kubernetes versions or managing compatibility
between different Kubernetes components can be challenging.
Solution: Follow the official Kubernetes documentation and release notes for
upgrade procedures. Use Kubernetes versioning tools like kubeadm or Kops for
simplified cluster upgrades. Validate compatibility between Kubernetes versions
and components before performing upgrades.
Follow
Written by Prateek Malhotra
170 followers · 17 following
I am Prateek Malhotra , a passionate DevOps Engineer with a deep love for implementing new technologies.
No responses yet
Mohit Salvi
Kubernetes Kubernetes Cluster K8s DevOps Linux
W h a t  a r e  y o u r  t h o u g h t s ?

page_label: 3
file_path: /content/articles/Troubleshooting Kubernetes performance issues in Production _ by Sobha _ Medium.pdf

Ensure that your pod specifications properly set resource requests and limits.
Step 7: Network and DNS Issues
7/11/25, 12:19 AM Troubleshooting Kubernetes performance issues in Production | by Sobha | Medium
https://medium.com/@sobha366/troubleshooting-kubernetes-performance-issues-in-production-d430d0a472b5 3/14

page_label: 4
file_path: /content/articles/Troubleshooting Kubernetes performance issues in Production _ by Sobha _ Medium.pdf

1. Network: Ensure pods can communicate. If pod-to-pod communication is
broken, check network policies:
kubectl describe networkpolicy -n <namespace>
2. DNS: If services are unreachable via DNS, verify that CoreDNS pods are healthy:
kubectl get pods -n kube-system | grep coredns
Check the CoreDNS logs for any errors:
kubectl logs <coredns_pod_name> -n kube-system
Step 8: Service and Endpoint Issues
1. Check the status of services:
kubectl get svc -n <namespace>
2. If a service is not routing traffic, describe it to check for configuration issues:
kubectl describe svc <service_name> -n <namespace>
3. Verify that the service has healthy endpoints:
kubectl get endpoints -n <namespace>
7/11/25, 12:19 AM Troubleshooting Kubernetes performance issues in Production | by Sobha | Medium
https://medium.com/@sobha366/troubleshooting-kubernetes-performance-issues-in-production-d430d0a472b5 4/14
---------------------
Given the context information and not prior knowledge, answer the query.
Query: What should I do before upgrading Kubernetes to a new version?
Answer:"""
}

# Clean the output field only
sample["output"] = clean_text(sample["output"])

# Result
print("Cleaned Output:\n")
print(sample["output"])

Cleaned Output:

---------------------
10. Upgrades and Compatibility:
Issue: Performing upgrades of Kubernetes versions or managing compatibility
between different Kubernetes components can be challenging.
Solution: Follow the official Kubernetes documentation and release notes for
upgrade procedures. Use Kubernetes versioning tools like kubeadm or Kops for
simplified cluster upgrades. Validate compatibility between Kubernetes versions
and components before performing upgrades.
Follow
Mohit Salvi
W h a t  a r e  y o u r  t h o u g h t s ?
Ensure that your pod specifications properly set resource requests and limits.
Step 7: Network and DNS Issues
1. Network: Ensure pods can communicate. If pod-to-pod communication is
broken, check network policies:
2. DNS: If services are unreachable via DNS, verify that CoreDNS pods are healthy:
kubectl get pods -n kube-system | grep coredns
Check the CoreDNS logs for any errors:
Step 8: Service and Endpoint Issues
1. Check the status of services:
2.

In [ ]:
print(documents)

[Document(id_='c0b210a0-6fce-48f6-99c1-888bc3c70b79', embedding=None, metadata={'page_label': '1', 'file_name': 'Top 10 Kubernetes Issues and Solutions_ Overcoming Common Challenges in Container Orchestration _ by Prateek Malhotra _ Medium.pdf', 'file_path': '/content/articles/Top 10 Kubernetes Issues and Solutions_ Overcoming Common Challenges in Container Orchestration _ by Prateek Malhotra _ Medium.pdf', 'file_type': 'application/pdf', 'file_size': 2422085, 'creation_date': '2025-07-29', 'last_modified_date': '2025-07-29'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='Top 10 Kubernetes Issues and Solutions:\nOvercoming Common Challen

In [ ]:
index = VectorStoreIndex.from_documents(documents)

In [ ]:
top_k = 3

# configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=top_k,
)

In [ ]:
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.5)],
)

In [ ]:
# query documents
query = "What should I do before upgrading Kubernetes to a new version?"
response = query_engine.query(query)

In [ ]:
# reformat response
context = "Context:\n"
for i in range(top_k):
    context = context + response.source_nodes[i].text + "\n\n"

print(context)

Context:
10. Upgrades and Compatibility:
Issue: Performing upgrades of Kubernetes versions or managing compatibility
between different Kubernetes components can be challenging.
Solution: Follow the official Kubernetes documentation and release notes for
upgrade procedures. Use Kubernetes versioning tools like kubeadm or Kops for
simplified cluster upgrades. Validate compatibility between Kubernetes versions
and components before performing upgrades.
Follow
Written by Prateek Malhotra
170 followers · 17 following
I am Prateek Malhotra , a passionate DevOps Engineer with a deep love for implementing new technologies.
No responses yet
Mohit Salvi
Kubernetes Kubernetes Cluster K8s DevOps Linux
W h a t  a r e  y o u r  t h o u g h t s ?

Ensure that your pod specifications properly set resource requests and limits.
Step 7: Network and DNS Issues
7/11/25, 12:19 AM Troubleshooting Kubernetes performance issues in Production | by Sobha | Medium
https://medium.com/@sobha366/troubleshooting-kube

In [ ]:
queries = [
    "How to fix networking issues in Kubernetes clusters?",
    "What causes persistent storage problems in Kubernetes?",
    "How do I scale Kubernetes pods and nodes efficiently?",
    "What is the best way to monitor logs in a Kubernetes environment?",
    "How to resolve resource allocation issues in Kubernetes?",
    "What are common service discovery and load balancing problems in Kubernetes?",
    "How to implement RBAC security in Kubernetes?",
    "How do I manage application deployments and rollbacks in Kubernetes?",
    "How to check the health of a Kubernetes cluster?",
    "What should I do before upgrading Kubernetes to a new version?"
]



In [ ]:
import json

rag_training_data = []

for query in queries:
    response = query_engine.query(query)

    rag_training_data.append({
        "instruction": query,
        "output": response.response.strip()
    })

# Save the collected data
with open("rag_finetune_data.json", "w", encoding="utf-8") as f:
    json.dump(rag_training_data, f, indent=2, ensure_ascii=False)

print("✅ RAG training data saved to rag_finetune_data.json")


✅ RAG training data saved to rag_finetune_data.json
